In [1]:
# === CELL 1 (NB4) — Reload/Resume ===
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = "/content/drive/MyDrive/FPX_Project"
sys.path.insert(0, os.path.join(DRIVE_ROOT, "utils"))
import checkpoint_utils as cu

config = cu.load_json("config", "project_config")
print("=== CELL 1 (NB4) === Resumed FPX project.")
print(f"Current phase: {config.get('current_phase')}")
cu.project_status()

Mounted at /content/drive
=== CELL 1 (NB4) === Resumed FPX project.
Current phase: P4_complete
=== FPX Project Drive Status ===
config: 2 file(s)
   - fault_config.json
   - project_config.json
data: 2 file(s)
   - client_partition.pkl
   - data_split.pkl
checkpoints/fl_sim: 21 file(s)
   - latest_round_state.pkl
   - round_001.pkl
   - round_002.pkl
   - round_003.pkl
   - round_004.pkl
   - round_005.pkl
   - round_006.pkl
   - round_007.pkl
   - round_008.pkl
   - round_009.pkl
checkpoints/signals: 2 file(s)
   - fault_config_used.json
   - signal_timeline.pkl
checkpoints/interface: 1 file(s)
   - generated_explanations.json
checkpoints/eval: 0 file(s)
outputs: 3 file(s)
   - fl_training_curve.png
   - p2_summary.json
   - signal_timeline.png
logs: 1 file(s)
   - project_log.txt


In [2]:
# === CELL 2 (NB4) — Load signal timeline from Notebook 3 ===
signal_history = cu.load_pickle("checkpoints/signals", "signal_timeline")
fault_config = cu.load_json("checkpoints/signals", "fault_config_used")

if signal_history is None:
    raise RuntimeError("Missing signal_timeline — run Notebook 3 first.")

print(f"=== CELL 2 (NB4) === Loaded {len(signal_history)} rounds of signal data.")
print(f"Fault config: {fault_config}")

=== CELL 2 (NB4) === Loaded 15 rounds of signal data.
Fault config: {'fault_client': 0, 'fault_round': 8, 'fault_type': 'label_shuffle'}


In [3]:
# === CELL 3 (NB4) — Explanation translation layer [FINAL v3] ===
explanation_utils_code = '''
def get_consecutive_excluded_count(history, round_idx, client_id):
    count = 0
    for i in range(round_idx, -1, -1):
        if history[i]["excluded"][client_id]:
            count += 1
        else:
            break
    return count

def get_notable_clients(record, z_threshold=-1.0, warmup_rounds=5):
    """Discover which clients look anomalous THIS round, purely from the signals.
    Mirrors the aggregation logic's warmup period: early rounds are known to be
    volatile before the federation stabilizes, so we do not narrate single-round
    noise during warmup as if it were a real signal - this keeps the explanation
    consistent with what the system actually acts on."""
    if record["round"] <= warmup_rounds:
        return []
    notable = []
    for i, z in enumerate(record["z_score"]):
        if z < z_threshold or record["excluded"][i]:
            notable.append(i)
    return notable

def explain_contribution(record):
    shares = record["effective_share"]
    max_i, min_i = shares.index(max(shares)), shares.index(min(shares))
    if max(shares) - min(shares) < 0.05:
        return "All sites contributed roughly equally to the global model this round."
    return (f"Site {max_i} contributed the most to this round's model update "
            f"({shares[max_i]*100:.0f}% effective weight), while Site {min_i} "
            f"contributed the least ({shares[min_i]*100:.0f}%).")

def explain_divergence(record, client_id, excluded_this_round):
    z = record["z_score"][client_id]
    if excluded_this_round:
        return (f"Site {client_id}'s update looked substantially different from the rest of "
                f"the federation this round (divergence score {z:.2f}, well outside the normal range).")
    else:
        return (f"Site {client_id}'s update has just started looking unusual compared to the rest of "
                f"the federation this round (divergence score {z:.2f}). Not yet acted on - "
                f"the system waits for a consistent pattern before down-weighting a site.")

def explain_exclusion(history, round_idx, client_id):
    record = history[round_idx]
    if not record["excluded"][client_id]:
        return None
    streak = get_consecutive_excluded_count(history, round_idx, client_id)
    return (f"Site {client_id}'s contribution was significantly down-weighted this round. "
            f"This site's updates have looked unusual for {streak} consecutive round(s) - "
            f"this may indicate a data quality or labeling issue at this site, and is worth investigating "
            f"before trusting its future contributions.")

def explain_instability(history, round_idx, drop_threshold=0.015):
    if round_idx == 0:
        return None
    delta = history[round_idx]["test_accuracy"] - history[round_idx-1]["test_accuracy"]
    if delta < -drop_threshold:
        anomalous = [i for i, z in enumerate(history[round_idx]["z_score"]) if z < -1.0]
        cause = f", coinciding with unusual updates from Site(s) {anomalous}" if anomalous else " with no single site clearly responsible"
        return f"Global accuracy dropped by {abs(delta)*100:.1f} points this round{cause}."
    return None

def generate_round_briefing(history, round_idx, z_threshold=-1.0, warmup_rounds=5):
    record = history[round_idx]

    lines = [f"--- Round {record['round']} briefing ---"]
    lines.append(f"Global test accuracy: {record['test_accuracy']*100:.1f}%")
    lines.append(explain_contribution(record))

    if record["round"] <= warmup_rounds:
        lines.append("Federation still stabilizing (warmup period) - anomaly detection not yet active.")
    else:
        notable = get_notable_clients(record, z_threshold=z_threshold, warmup_rounds=warmup_rounds)
        if not notable:
            lines.append("No sites showed unusual update behavior this round.")
        else:
            for cid in notable:
                lines.append(explain_divergence(record, cid, record["excluded"][cid]))
                excl_text = explain_exclusion(history, round_idx, cid)
                if excl_text:
                    lines.append(excl_text)

    instab_text = explain_instability(history, round_idx)
    if instab_text:
        lines.append(instab_text)

    return "\\n".join(lines)
'''

util_path = os.path.join(DRIVE_ROOT, "utils", "explanation_utils.py")
with open(util_path, "w") as f:
    f.write(explanation_utils_code)

sys.path.insert(0, os.path.join(DRIVE_ROOT, "utils"))
import importlib
if "explanation_utils" in sys.modules:
    importlib.reload(sys.modules["explanation_utils"])
import explanation_utils as eu

print("=== CELL 3 (NB4) === explanation_utils.py finalized (warmup-aware) and reloaded.")

=== CELL 3 (NB4) === explanation_utils.py finalized (warmup-aware) and reloaded.


In [4]:
# === CELL 4 (NB4) — Generate full explanation set [PATCHED] ===
all_briefings = []
for idx in range(len(signal_history)):
    record = signal_history[idx]
    briefing_text = eu.generate_round_briefing(signal_history, idx)
    all_briefings.append({
        "round": record["round"],
        "briefing_text": briefing_text,
        "is_fault_round": record.get("is_fault_round", False),
        "notable_clients": eu.get_notable_clients(record)
    })

cu.save_json(all_briefings, "checkpoints/interface", "generated_explanations")
print(f"=== CELL 4 (NB4) === Generated and saved {len(all_briefings)} round briefings.")

[2026-09-03 06:39:51] Saved JSON checkpoint: checkpoints/interface/generated_explanations.json
=== CELL 4 (NB4) === Generated and saved 15 round briefings.


In [6]:
# === CELL 5 (NB4) — Preview key rounds ===
key_rounds = [3, fault_config["fault_round"], fault_config["fault_round"] + 1, len(signal_history)]

for r in key_rounds:
    if 1 <= r <= len(all_briefings):
        print(all_briefings[r-1]["briefing_text"])
        print()

config["current_phase"] = "P4_complete"
cu.save_json(config, "config", "project_config")
print("=== CELL 5 (NB4) === Preview complete. current_phase set to P4_complete.")

--- Round 3 briefing ---
Global test accuracy: 73.7%
Site 3 contributed the most to this round's model update (20% effective weight), while Site 0 contributed the least (4%).
Federation still stabilizing (warmup period) - anomaly detection not yet active.

--- Round 8 briefing ---
Global test accuracy: 93.0%
Site 3 contributed the most to this round's model update (20% effective weight), while Site 0 contributed the least (4%).
Site 0's update has just started looking unusual compared to the rest of the federation this round (divergence score -1.85). Not yet acted on - the system waits for a consistent pattern before down-weighting a site.

--- Round 9 briefing ---
Global test accuracy: 94.7%
Site 3 contributed the most to this round's model update (21% effective weight), while Site 0 contributed the least (0%).
Site 0's update looked substantially different from the rest of the federation this round (divergence score -2.15, well outside the normal range).
Site 0's contribution was sig